# ECCOFS `avg` temp: read performance of three chunk layouts

Compares the same reads against three stores of the same data (166 times × 50 levels × 1443 × 1667, float32):

| name | where | chunks (time, s_rho, eta, xi) |
|---|---|---|
| `original` | virtual Icechunk over NOAA NetCDF (`rsignell/eccofs-avg-best`) | 1, 1, 145, 167 (~0.1 MB) |
| `balanced` | native Icechunk (`rsignell/eccofs-avg-temp-balanced-10mb`) | 17, 5, 144, 167 (~8 MB) |
| `map` | native Icechunk (`rsignell/eccofs-avg-temp-map-10mb`) | 3, 50, 120, 139 (~10 MB) |

The rechunked stores were written by `rechunk_vars.py` with chunk shapes from `dynamic_chunks.iterative_ratio_increase_algo` (10 MB target). Everything is read anonymously.

Timings depend on where this runs (network distance to NOAA's us-east-1 bucket and to OSN). Each test is repeated at different times/places to avoid cache hits, and the median is reported.

In [ ]:
import time
import icechunk
import numpy as np
import pandas as pd
import xarray as xr
import zarr

zarr.config.set({"async.concurrency": 128})

ENDPOINT, BUCKET = "https://usgs.osn.mghpcc.org", "esip"
SOURCE = "s3://noaa-nos-eccofs-pds/"


def native(prefix):
    st = icechunk.s3_storage(bucket=BUCKET, prefix=prefix, region="us-east-1", endpoint_url=ENDPOINT,
                             anonymous=True, force_path_style=True)
    return icechunk.Repository.open(st)


def virtual(prefix):
    st = icechunk.s3_storage(bucket=BUCKET, prefix=prefix, region="us-east-1", endpoint_url=ENDPOINT,
                             anonymous=True, force_path_style=True)
    cfg = icechunk.RepositoryConfig.default()
    cfg.set_virtual_chunk_container(
        icechunk.VirtualChunkContainer(SOURCE, icechunk.s3_store(region="us-east-1", anonymous=True)))
    return icechunk.Repository.open(
        st, cfg, authorize_virtual_chunk_access=icechunk.containers_credentials(
            {SOURCE: icechunk.s3_anonymous_credentials()}))


repos = {
    "original": virtual("rsignell/eccofs-avg-best"),
    "balanced": native("rsignell/eccofs-avg-temp-balanced-10mb"),
    "map": native("rsignell/eccofs-avg-temp-map-10mb"),
}
temp = {k: xr.open_zarr(r.readonly_session("main").store, consolidated=False, chunks={}).temp
        for k, r in repos.items()}
for k, v in temp.items():
    print(k, dict(v.sizes), v.encoding["chunks"])

## Tests

Each test is a function of (array, repeat index) so repeats hit different times/locations. Points are ocean cells (not NaN at the surface).

In [ ]:
sfc = temp["original"].isel(time=0, s_rho=-1).load()
ocean_j, ocean_i = np.where(~np.isnan(sfc.values))
rng = np.random.default_rng(0)
pick = rng.choice(len(ocean_j), 3, replace=False)
POINTS = [(int(ocean_j[k]), int(ocean_i[k])) for k in pick]
TIMES = [20, 80, 140]
print(POINTS)

tests = {
    "surface map (1 time)":
        lambda a, r: a.isel(time=TIMES[r], s_rho=-1),
    "full-depth map (1 time, 50 levels)":
        lambda a, r: a.isel(time=TIMES[r]),
    "surface point series (all times)":
        lambda a, r: a.isel(s_rho=-1, eta_rho=POINTS[r][0], xi_rho=POINTS[r][1]),
    "point profile (1 time, all levels)":
        lambda a, r: a.isel(time=TIMES[r], eta_rho=POINTS[r][0], xi_rho=POINTS[r][1]),
    "point, all levels, 20 times":
        lambda a, r: a.isel(time=slice(TIMES[r], TIMES[r] + 20), eta_rho=POINTS[r][0], xi_rho=POINTS[r][1]),
}

In [ ]:
rows = []
for test, fn in tests.items():
    for name, arr in temp.items():
        for r in range(3):
            sel = fn(arr, r)
            t0 = time.perf_counter()
            out = sel.load()
            rows.append(dict(test=test, layout=name, repeat=r, seconds=time.perf_counter() - t0, MB=out.nbytes / 1e6))
        print(test, name, "done", flush=True)
res = pd.DataFrame(rows)

## Results (median of 3 repeats, seconds)

In [ ]:
table = res.groupby(["test", "layout"]).seconds.median().unstack("layout")[list(temp)]
table = table.loc[list(tests)]
table.round(2)

In [ ]:
ax = table.plot.barh(figsize=(9, 5), width=0.8)
ax.invert_yaxis(); ax.set_xlabel("seconds (median of 3)"); ax.set_ylabel("")
ax.set_title("ECCOFS temp read time by chunk layout"); ax.grid(axis="x", alpha=.3)